# Naloxone related posts on Reddit

This notebook reuses the subreddit archive from `reddit_user_representativeness.ipynb` to measure how often naloxone (also known as Narcan) appears in the selected communities and to explore how people talk about it.

In [7]:
import json
import os
import re
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm
import zstandard

pd.set_option('display.max_columns', 20)


In [8]:
subreddits = ['addiction', 'fentanyl', 'Drugs', 'FentanylRecovery', 'opiates', 'OpiatesRecovery']
input_dir = Path('../../reddit/subreddits24')
start_time = datetime(2021, 1, 1, tzinfo=timezone.utc).timestamp()

keyword_terms = [
    'naloxone',
    'narcan',
]
escaped_keywords = sorted((re.escape(term) for term in keyword_terms), key=len, reverse=True)
keyword_pattern = re.compile(r"\b(?:" + "|".join(escaped_keywords) + r")\b", re.IGNORECASE)

analysis_timezone = timezone.utc

input_dir.exists()


True

In [9]:
def read_and_decode(reader, chunk_size, max_window_size, previous_chunk=None, bytes_read=0):
    chunk = reader.read(chunk_size)
    bytes_read += chunk_size
    if previous_chunk is not None:
        chunk = previous_chunk + chunk
    try:
        return chunk.decode()
    except UnicodeDecodeError:
        if bytes_read > max_window_size:
            raise UnicodeError(f'Unable to decode frame after reading {bytes_read:,} bytes')
        return read_and_decode(reader, chunk_size, max_window_size, chunk, bytes_read)


def read_lines_zst(file_name):
    with open(file_name, 'rb') as file_handle:
        buffer = ''
        reader = zstandard.ZstdDecompressor(max_window_size=2**31).stream_reader(file_handle)
        while True:
            chunk = read_and_decode(reader, 2**27, (2**29) * 2)
            if not chunk:
                break
            lines = (buffer + chunk).split('\n')
            for line in lines[:-1]:
                yield line, file_handle.tell()
            buffer = lines[-1]
        reader.close()

def extract_text(obj, kind):
    if kind == 'submissions':
        # Keep submission text as selftext only.
        # Title is stored separately and can be modeled as another field.
        selftext = obj.get('selftext') or ''
        if selftext in {'[removed]', '[deleted]'}:
            selftext = ''
        text = selftext
    else:
        body = obj.get('body') or ''
        if body in {'[removed]', '[deleted]'}:
            body = ''
        text = body
    return text

def clean_text(text):
    return re.sub(r'\s+', ' ', text or '').strip()

def month_bucket(ts):
    return datetime.fromtimestamp(ts, tz=analysis_timezone).strftime('%Y-%m')



In [10]:
mention_records = []

for subreddit in subreddits:
    for kind in ('submissions', 'comments'):
        file_path = input_dir / f'{subreddit}_{kind}.zst'
        if not file_path.exists():
            print(f'Missing file: {file_path}')
            continue
        for line, _ in tqdm(read_lines_zst(str(file_path)), desc=f'{subreddit}-{kind}', unit='rows'):
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            created_utc = obj.get('created_utc')
            if created_utc is None:
                continue
            created_ts = int(created_utc)
            if created_ts < start_time:
                continue

            text = extract_text(obj, kind)
            if not text:
                continue

            matches = keyword_pattern.findall(text)
            if not matches:
                continue

            record = {
                'subreddit': subreddit,
                'kind': kind,
                'created_utc': created_ts,
                'created_dt': datetime.fromtimestamp(created_ts, tz=analysis_timezone),
                'month': month_bucket(created_ts),
                'mention_count': len(matches),
                'keyword_hits': sorted({m.lower() for m in matches}),
                'score': obj.get('score'),
                'author': obj.get('author'),
                'id': obj.get('id'),
                'text': clean_text(text),
            }
            if kind == 'submissions':
                record['title'] = obj.get('title') or ''
                record['num_comments'] = obj.get('num_comments')
                record['permalink'] = obj.get('permalink')
            else:
                record['parent_id'] = obj.get('parent_id')
                record['link_id'] = obj.get('link_id')
            mention_records.append(record)

print(f'Total naloxone mentions collected: {len(mention_records):,}')


addiction-submissions: 81516rows [00:02, 33331.07rows/s]
addiction-comments: 531592rows [00:08, 65935.08rows/s] 
fentanyl-submissions: 26650rows [00:00, 41615.49rows/s]
fentanyl-comments: 284754rows [00:04, 65973.20rows/s]
Drugs-submissions: 1099233rows [00:18, 59204.02rows/s]
Drugs-comments: 12692928rows [02:00, 105037.11rows/s]
FentanylRecovery-submissions: 3229rows [00:00, 28653.53rows/s]
FentanylRecovery-comments: 32434rows [00:00, 56291.21rows/s]
opiates-submissions: 368904rows [00:06, 58314.72rows/s] 
opiates-comments: 5184595rows [00:47, 108873.13rows/s]
OpiatesRecovery-submissions: 62299rows [00:01, 40005.90rows/s]
OpiatesRecovery-comments: 787052rows [00:09, 81808.58rows/s] 

Total naloxone mentions collected: 29,867


In [11]:
if mention_records:
    mentions_df = pd.DataFrame(mention_records)
    mentions_df['created_dt'] = pd.to_datetime(mentions_df['created_dt'])
else:
    mentions_df = pd.DataFrame(
        columns=['subreddit', 'kind', 'created_utc', 'created_dt', 'month', 'mention_count', 'keyword_hits', 'score', 'author', 'id', 'text']
    )

# 保存到文件
output_file = 'reddit_keyword_mentions.csv'
mentions_df.to_csv(output_file, index=False)
print(f'Saved {len(mentions_df):,} mentions to {output_file}')

# 显示前几条
display(mentions_df.head())


Saved 29,867 mentions to reddit_keyword_mentions.csv


,subreddit,kind,created_utc,created_dt,month,mention_count,keyword_hits,score,author,id,text,title,num_comments,permalink,parent_id,link_id
0,addiction,submissions,1610710173,2021-01-15 11:29:33+00:00,2021-01,1,[narcan],3,FoxyFreckles1989,kxsrbs,I am 100% for decriminalizing all illicit subs...,"I’d like to talk about treatment, harm reducti...",8.0,/r/addiction/comments/kxsrbs/id_like_to_talk_a...,NaN,NaN
1,addiction,submissions,1611346445,2021-01-22 20:14:05+00:00,2021-01,1,[narcan],58,[deleted],l2vzav,"Not sure is this allowed here, but I can’t see...",Boyfriend overdosed in bathroom,38.0,/r/addiction/comments/l2vzav/boyfriend_overdos...,NaN,NaN
2,addiction,submissions,1611873309,2021-01-28 22:35:09+00:00,2021-01,1,[narcan],15,killemkiller,l7b3du,...Coming to a city near you... -Welcome to th...,For what it's worth..,8.0,/r/addiction/comments/l7b3du/for_what_its_worth/,NaN,NaN
3,addiction,submissions,1612319109,2021-02-03 02:25:09+00:00,2021-02,1,[narcan],10,SoSoberMom,lbd8zk,Through the eyes of a formerly addicted mother...,Heroin Is Stronger Than A Mother's Love,3.0,/r/addiction/comments/lbd8zk/heroin_is_stronge...,NaN,NaN
4,addiction,submissions,1613678132,2021-02-18 19:55:32+00:00,2021-02,2,"[naloxone, narcan]",1,Low-Grocery5490,lmvlyu,"My organization, Project Opioid, based in Cent...",Seeking your insights to help my org build a r...,0.0,/r/addiction/comments/lmvlyu/seeking_your_insi...,NaN,NaN


In [12]:
# 在notebook中运行
print("数据质量分析：")
print(f"总数: {len(mentions_df):,}")
print(f"\n按长度分布:")
print(mentions_df['text'].str.len().describe())
print(f"\n按分数分布:")
print(mentions_df['score'].describe())
print(f"\n按subreddit分布:")
print(mentions_df['subreddit'].value_counts())

数据质量分析：
总数: 29,867

按长度分布:
count    29867.000000
mean       669.765025
std       1084.681920
min          6.000000
25%        175.000000
50%        358.000000
75%        754.000000
max      39740.000000
Name: text, dtype: float64

按分数分布:
count    29867.000000
mean         5.783641
std         41.305692
min        -70.000000
25%          1.000000
50%          1.000000
75%          3.000000
max       2606.000000
Name: score, dtype: float64

按subreddit分布:
subreddit
opiates             13726
Drugs                8958
fentanyl             3441
OpiatesRecovery      2199
addiction            1072
FentanylRecovery      471
Name: count, dtype: int64
